# Promote, reject, and roll back model versions

**Goal.** Work through a bounded, reproducible example and inspect the evidence before connecting an external service.

**Prerequisites.** Base FraudTwin install. Optional extras and Docker commands are clearly marked.

**Produces.** Tables, fingerprints, manifests, and verification output.


**Source size.** The default cells generate approximately 1,000 logical payments; increase duration and population together for a 10,000-payment run.

**Offline path.** All marked offline cells run without Docker or network services. Service cells are optional and explicitly marked in notebook metadata.

**Cleanup.** Outputs are written under a temporary directory; remove any local run directory if you changed the output location.


In [1]:
!pip install mlflow
import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"
print("MLflow tracking URI configured: http://127.0.0.1:5000")

MLflow tracking URI configured: http://127.0.0.1:5000


## Optional MLflow setup
The notebook works without a server by using a temporary local tracking store. To make the run appear in the MLflow web UI, start the tracking server from a separate terminal in the repository root and keep that terminal running:

```bash
poetry install -E mlflow
poetry run mlflow server --host 127.0.0.1 --port 5000
```

Then open **http://127.0.0.1:5000** in the same machine's browser. `localhost:5000` works too, but only while the command above is running; opening the URL alone cannot start MLflow. If the notebook runs in Docker, WSL, or a remote Jupyter server, use the forwarded/public host and port instead of your browser's localhost.

After the server is running, execute the first optional MLflow setup cell below. It sets the tracking URI in the Jupyter kernel. Then execute the MLflow tracking cell near the end of the notebook; merely reading this Markdown code block does not configure the kernel:

```python
import os
os.environ["MLFLOW_TRACKING_URI"] = "http://127.0.0.1:5000"
```

If the setup cell is skipped, the notebook intentionally uses a temporary local store and the browser UI will remain empty. This tutorial logs a standard MLflow tracking run, not a GenAI trace. In MLflow 3, choose the **Model training** tab at the top of the experiment page; the **GenAI** overview will correctly show zero traces. If the URL contains `workflowType=genai`, remove that query parameter or use the direct run link printed by the tracking cell. See the [MLflow tracking server guide](https://mlflow.org/docs/latest/self-hosting/architecture/tracking-server/) for server and client configuration details.


**Set up a deterministic source run**


In [2]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import polars as pl

from fraudtwin.config import load_config
from fraudtwin.generation import generate

root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs" / "minimal.yaml").exists()),
    Path.cwd(),
)
base = load_config(root / "configs" / "minimal.yaml")
# Scale the population so the bounded example produces about 1,000 payments.
population = base.population.model_copy(
    update={
        "customers": 200,
        "accounts": 300,
        "cards": 240,
        "devices": 240,
        "pix_keys": 160,
        "merchants": 60,
    }
)
simulation = base.simulation.model_copy(update={"duration_days": 10})
fraud = base.fraud.model_copy(update={"enabled": True, "target_rate": 0.05})
config = base.model_copy(
    update={"population": population, "simulation": simulation, "fraud": fraud}
)
data = generate(config, write=False)
run_id = data.run_id
payments = pl.DataFrame([item.model_dump(mode="json") for item in data.behavior.payments])
print("Generated source run")
print(f"  run id: {run_id}")
print(f"  payments: {len(payments):,}")
print(f"  events: {len(data.behavior.payment_events):,}")

Generated source run
  run id: RUN-2a3ad02ee370aeb8
  payments: 1,092
  events: 4,699


**Score a small evaluation sample**


In [3]:
from fraudtwin.ml import heuristic_predictions

dataset = data.require_dataset()
policy_rows = dataset.rows[:20]
scores = heuristic_predictions(policy_rows)
metrics = {"rows": len(scores), "mean_score": sum(s.fraud_score for s in scores) / len(scores)}
print("Candidate metrics")
print(f"  rows scored: {metrics['rows']:,}")
print(f"  mean fraud score: {metrics['mean_score']:.3f}")

Candidate metrics
  rows scored: 20
  mean fraud score: 0.245


**Create a candidate model record**


In [4]:
candidate = {"version": "candidate-1", "schema": "pit-v1", "metrics": metrics}
display(pl.DataFrame([
    {"version": candidate["version"], "schema": candidate["schema"], **candidate["metrics"]}
]))

version,schema,rows,mean_score
str,str,i64,f64
"""candidate-1""","""pit-v1""",20,0.244821


**Promote the candidate**


In [5]:
registry = {"Production": None, "Staging": candidate}
registry["Production"] = registry["Staging"]
print("Promotion")
print(f"  staging -> production: {registry['Production']['version']}")

Promotion
  staging -> production: candidate-1


**Roll back to the previous version**


In [6]:
registry["Staging"] = {**candidate, "version": "candidate-2"}
registry["Production"] = candidate
print("Rollback")
print(f"  production restored to: {registry['Production']['version']}")

Rollback
  production restored to: candidate-1


**Verify the promotion contract**


In [7]:
assert registry["Production"]["schema"] == "pit-v1"
print("Optional MLflow registry calls can replace this local state machine.")

Optional MLflow registry calls can replace this local state machine.


**Verify invariants and clean up**


In [8]:
# A compact inspection is more useful than printing an entire run.
sample_columns = [
    c
    for c in ("payment_id", "amount", "initiated_at", "payer_account_id")
    if c in payments.columns
]
sample_rows = payments.select(sample_columns).head(8).to_dicts()
print(f"Sample payments ({len(sample_rows)} of {payments.height} rows):")
for row in sample_rows:
    print(
        f"  - {row.get('payment_id')}: amount={row.get('amount')}, "
        f"initiated_at={row.get('initiated_at')}, payer={row.get('payer_account_id')}"
    )
nulls = {
    name: count
    for name, count in payments.null_count().to_dicts()[0].items()
    if count
}
print("\nData quality summary:")
print(f"  rows: {payments.height}")
print(f"  columns: {payments.width}")
if not nulls:
    print("  nulls: none")
else:
    print("  columns with nulls:")
    for name, count in sorted(nulls.items()):
        print(f"    - {name}: {count}")

Sample payments (8 of 1092 rows):
  - PAY-00000001: amount=70.7, initiated_at=2026-01-03T16:25:00Z, payer=ACC-000123
  - PAY-00000002: amount=25.52, initiated_at=2026-01-05T11:37:00Z, payer=ACC-000174
  - PAY-00000003: amount=18.37, initiated_at=2026-01-05T11:21:00Z, payer=ACC-000174
  - PAY-00000004: amount=5.54, initiated_at=2026-01-02T22:30:00Z, payer=ACC-000003
  - PAY-00000005: amount=13.71, initiated_at=2026-01-02T09:41:00Z, payer=ACC-000029
  - PAY-00000006: amount=70.06, initiated_at=2026-01-04T10:40:00Z, payer=ACC-000179
  - PAY-00000007: amount=42.73, initiated_at=2026-01-02T18:04:00Z, payer=ACC-000247
  - PAY-00000008: amount=36.42, initiated_at=2026-01-05T09:27:00Z, payer=ACC-000255

Data quality summary:
  rows: 1092
  columns: 15
  columns with nulls:
    - card_id: 565
    - merchant_id: 565
    - payee_account_id: 86
    - payee_institution_id: 86
    - payee_pix_key_id: 857
    - payer_institution_id: 86
    - payer_pix_key_id: 857


**Optional service integration**


In [9]:
summary = {
    "run_id": run_id,
    "payments": len(data.behavior.payments),
    "payment_events": len(data.behavior.payment_events),
    "fraud_records": len(data.behavior.fraud_records),
}
assert summary["payments"] == len(payments)
assert summary["payments"] > 0
print("Generated dataset")
print(f"  run id: {summary['run_id']}")
print(f"  payments: {summary['payments']:,}")
print(f"  payment events: {summary['payment_events']:,}")
print(f"  fraud records: {summary['fraud_records']:,}")

Generated dataset
  run id: RUN-2a3ad02ee370aeb8
  payments: 1,092
  payment events: 4,699
  fraud records: 51


**Review the expected outcome**


In [10]:
import os
from urllib.parse import urlparse
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")

try:
    import mlflow
    from IPython.display import Markdown, display
    from mlflow.tracking import MlflowClient

    tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
    with TemporaryDirectory(prefix="fraudtwin-mlflow-") as mlflow_dir:
        mlflow.set_tracking_uri(tracking_uri or Path(mlflow_dir).as_uri())
        mlflow.set_experiment("fraudtwin-tutorial")
        with mlflow.start_run() as run:
            mlflow.log_params({"schema": candidate["schema"]})
            mlflow.log_metric("mean_score", metrics["mean_score"])
            artifact = Path(mlflow_dir) / "candidate.json"
            artifact.write_text(json.dumps(candidate), encoding="utf-8")
            mlflow.log_artifact(str(artifact))
            tracked_id = run.info.run_id
        client = MlflowClient()
        experiment = client.get_experiment_by_name("fraudtwin-tutorial")
        tracked = client.get_run(tracked_id)
        experiments = client.search_experiments()
        print("MLflow tracking")
        print(f"  tracking URI: {mlflow.get_tracking_uri()}")
        print(f"  experiment: {experiment.name} (id={experiment.experiment_id})")
        display(
            pl.DataFrame(
                [{"experiment": item.name, "experiment_id": item.experiment_id} for item in experiments]
            )
        )
        print("  tracked: yes")
        print(f"  run id: {tracked.info.run_id}")
        print(f"  metrics logged: {len(tracked.data.metrics)}")
        if urlparse(tracking_uri or "").scheme in {"http", "https"}:
            run_url = (
                f"{tracking_uri.rstrip('/')}/#/experiments/"
                f"{tracked.info.experiment_id}/runs/{tracked.info.run_id}"
            )
            display(Markdown(f"[Open this run in MLflow]({run_url})"))
        else:
            print("  browser link: unavailable for the temporary local store")
except Exception as exc:
    print("MLflow tracking is unavailable; the offline promotion state remains valid.")
    print("  tracked: no")
    print("  offline_fallback: yes")
    print(f"  reason: {type(exc).__name__}")

🏃 View run angry-shoat-568 at: http://127.0.0.1:5000/#/experiments/1/runs/bdb270b811f3440d8afaa0ae041ddd01
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
MLflow tracking
  tracking URI: http://127.0.0.1:5000
  experiment: fraudtwin-tutorial (id=1)


experiment,experiment_id
str,str
"""fraudtwin-tutorial""","""1"""
"""Default""","""0"""


  tracked: yes
  run id: bdb270b811f3440d8afaa0ae041ddd01
  metrics logged: 1


[Open this run in MLflow](http://127.0.0.1:5000/#/experiments/1/runs/bdb270b811f3440d8afaa0ae041ddd01)

## Record the generated shape and tutorial contract.


In [11]:
summary = {
    "payments": len(data.behavior.payments),
    "events": len(data.behavior.payment_events),
}
print("Tutorial contract")
print(f"  payments: {summary['payments']:,}")
print(f"  events: {summary['events']:,}")
assert summary["payments"] >= 0

Tutorial contract
  payments: 1,092
  events: 4,699


In [12]:
assert registry["Production"]["schema"] == "pit-v1"
print("Offline promotion fallback")
print("  offline_fallback: yes")
print(f"  production version: {registry['Production']['version']}")

Offline promotion fallback
  offline_fallback: yes
  production version: candidate-1
